# Analytics Case Study — Loyalty Points Analysis

In [1]:
import pandas as pd

# Part A — Calculating Loyalty Points

In [11]:
file = "Analytics Position Case Study.xlsx"

gameplay = pd.read_excel(file, sheet_name="User Gameplay data", header=3)
deposit = pd.read_excel(file, sheet_name="Deposit Data", header=3)
withdrawal = pd.read_excel(file, sheet_name="Withdrawal Data", header=3)

In [12]:
gameplay.columns = gameplay.columns.str.strip()
deposit.columns = deposit.columns.str.strip()
withdrawal.columns = withdrawal.columns.str.strip()

In [13]:
gameplay.columns = gameplay.columns.str.lower()
deposit.columns = deposit.columns.str.lower()
withdrawal.columns = withdrawal.columns.str.lower()

In [15]:
print(gameplay.columns.tolist())
print(deposit.columns.tolist())
print(withdrawal.columns.tolist())

['user id', 'games played', 'datetime']
['user id', 'datetime', 'amount']
['user id', 'datetime', 'amount']


In [18]:
gameplay["datetime"] = pd.to_datetime(gameplay["datetime"])
deposit["datetime"] = pd.to_datetime(deposit["datetime"])
withdrawal["datetime"] = pd.to_datetime(withdrawal["datetime"])

In [40]:
# Filter only October 2022 data
gameplay = gameplay[
    (gameplay["datetime"].dt.month == 10) &
    (gameplay["datetime"].dt.year == 2022)
]

deposit = deposit[
    (deposit["datetime"].dt.month == 10) &
    (deposit["datetime"].dt.year == 2022)
]

withdrawal = withdrawal[
    (withdrawal["datetime"].dt.month == 10) &
    (withdrawal["datetime"].dt.year == 2022)
]

In [19]:
gameplay["date"] = gameplay["datetime"].dt.date
deposit["date"] = deposit["datetime"].dt.date
withdrawal["date"] = withdrawal["datetime"].dt.date

## Step 1 — Create time slots (S1 and S2)

S1 → 12am to 12pm  
S2 → 12pm to 12am

In [20]:
def get_slot(x):
    if x.hour < 12:
        return "S1"
    else:
        return "S2"

gameplay["slot"] = gameplay["datetime"].apply(get_slot)
deposit["slot"] = deposit["datetime"].apply(get_slot)
withdrawal["slot"] = withdrawal["datetime"].apply(get_slot)

## Step 2 — Aggregate player activity per day and slot

Calculate:
• games played
• deposit amount & count
• withdrawal amount & count

In [21]:
games_summary = gameplay.groupby(
    ["user id","date","slot"]
)["games played"].sum().reset_index()

In [22]:
deposit_summary = deposit.groupby(
    ["user id","date","slot"]
).agg(
    deposit_amount=("amount","sum"),
    num_deposits=("amount","count")
).reset_index()

In [23]:
withdraw_summary = withdrawal.groupby(
    ["user id","date","slot"]
).agg(
    withdrawal_amount=("amount","sum"),
    num_withdrawals=("amount","count")
).reset_index()

In [24]:
merged = games_summary.merge(
    deposit_summary,
    on=["user id","date","slot"],
    how="outer"
)

merged = merged.merge(
    withdraw_summary,
    on=["user id","date","slot"],
    how="outer"
)

merged = merged.fillna(0)

## Step 3 — Calculate loyalty points

Formula used:

Loyalty Points =
0.01 × Deposit Amount  
+ 0.005 × Withdrawal Amount  
+ 0.001 × (Deposits − Withdrawals)  
+ 0.2 × Games Played

In [38]:
merged["loyalty points"] = (
    0.01 * merged["deposit_amount"]
    + 0.005 * merged["withdrawal_amount"]
    + 0.001 * (merged["num_deposits"] - merged["num_withdrawals"]).clip(lower=0)
    + 0.2 * merged["games played"]
)

## Display players ranked by loyalty points for each required slot.

In [37]:
slot_results = slot_results.sort_values(
    ["date","slot","loyalty points"],
    ascending=[True,True,False]
)

slot_results = slot_results.reset_index(drop=True)

slot_results

,user id,date,slot,games played,deposit_amount,num_deposits,withdrawal_amount,num_withdrawals,loyalty points
0,634,2022-10-16,S2,0.0,0.0,0.0,298311.0,1.0,1491.555
1,212,2022-10-16,S2,0.0,99999.0,1.0,0.0,0.0,999.991
2,99,2022-10-16,S2,0.0,98000.0,2.0,0.0,0.0,980.002
3,28,2022-10-16,S2,0.0,90000.0,4.0,0.0,0.0,900.004
4,566,2022-10-16,S2,1.0,88000.0,3.0,0.0,0.0,880.203
...,...,...,...,...,...,...,...,...,...
1839,944,2022-10-26,S2,1.0,0.0,0.0,0.0,0.0,0.200
1840,959,2022-10-26,S2,1.0,0.0,0.0,0.0,0.0,0.200
1841,991,2022-10-26,S2,1.0,0.0,0.0,0.0,0.0,0.200
1842,995,2022-10-26,S2,1.0,0.0,0.0,0.0,0.0,0.200


## Q2 — Monthly leaderboard ranking (October)

Rank players based on:

1️⃣ Total loyalty points  
2️⃣ Total games played (tie-breaker)

In [28]:
monthly = merged.groupby("user id").agg(
    total_points=("loyalty points","sum"),
    total_games=("games played","sum"),
    total_deposit=("deposit_amount","sum")
).reset_index()

monthly = monthly.sort_values(
    by=["total_points","total_games"],
    ascending=False
)

monthly.head(10)

,user id,total_points,total_games,total_deposit
634,634,83843.326,24.0,515000.0
99,99,23665.744,10.0,1164800.0
672,672,22757.783,10.0,2158700.0
212,212,22199.286,1.0,1924981.0
740,740,19211.826,2.0,1738490.0
566,566,19153.757,183.0,1819175.0
714,714,16764.234,6.0,1676300.0
421,421,15446.492,1557.0,878600.0
369,369,14438.451,37.0,650000.0
30,30,14053.376,13.0,1329000.0


## Q3, Q4, Q5 — Average player metrics

Calculate:

• Average deposit amount  
• Average deposit per user  
• Average games played per user

In [34]:
print("Average deposit amount:", round(avg_deposit,2))

Average deposit amount: 5492.19


In [35]:
print("Average deposit per user:", round(avg_deposit_per_user,2))


Average deposit per user: 104669.65


In [36]:
print("Average games per user:", round(avg_games_per_user,2))

Average games per user: 355.27


# Part B — Bonus allocation for top 50 players

Bonus pool = ₹50,000

Bonus distributed proportionally based on total loyalty points.

In [32]:
top50 = monthly.head(50).copy()

total_points_top50 = top50["total_points"].sum()

bonus_pool = 50000

top50["bonus"] = (
    top50["total_points"] / total_points_top50
) * bonus_pool

top50

,user id,total_points,total_games,total_deposit,bonus
634,634,83843.326,24.0,515000.0,6638.859248
99,99,23665.744,10.0,1164800.0,1873.894452
672,672,22757.783,10.0,2158700.0,1802.000533
212,212,22199.286,1.0,1924981.0,1757.777777
740,740,19211.826,2.0,1738490.0,1521.225538
566,566,19153.757,183.0,1819175.0,1516.627535
714,714,16764.234,6.0,1676300.0,1327.420980
421,421,15446.492,1557.0,878600.0,1223.079894
369,369,14438.451,37.0,650000.0,1143.261468
30,30,14053.376,13.0,1329000.0,1112.770565


In [33]:
slot_results.to_excel("slot_results.xlsx", index=False)

monthly.to_excel("monthly_leaderboard.xlsx", index=False)

top50.to_excel("top50_bonus.xlsx", index=False)

# Part C — Improved Loyalty Formula

## Objective
Evaluate the fairness of the existing loyalty points formula and propose an improved formula that better balances financial contribution and gameplay engagement.

---

## 1. Calculate active days per user

Active days represent the number of unique days a player was active on the platform.  
This metric helps measure consistency of user engagement rather than only transaction value.

Active days are calculated as the count of unique dates for each user.

---

## 2. New improved loyalty formula

The existing formula gives high importance to deposit and withdrawal amounts, which may favour high-spending users even if gameplay activity is low.

To better reflect user engagement, the improved formula:

• reduces the weight of withdrawals  
• increases importance of gameplay activity  
• rewards consistent platform usage  
• considers frequency of transactions  

### Proposed Formula

New Loyalty Points =

(0.01 × Deposit Amount)  
+ (0.002 × Withdrawal Amount)  
+ (0.01 × max(Number of Deposits − Number of Withdrawals, 0))  
+ (0.3 × Number of Games Played)  
+ (2 × Active Days)

---

## 3. Monthly ranking using new formula

Players are ranked based on total loyalty points calculated using the improved formula.

Ranking criteria:

1. Higher total loyalty points rank higher
2. In case of tie, player with more games played ranks higher

---

## 4. Comparison of old vs new ranking

Comparison between the old and new formulas shows:

• players with consistent gameplay move higher in ranking  
• players with very high deposits but low gameplay may move slightly lower  
• overall ranking becomes more balanced between spending and engagement  

The improved formula better represents true player loyalty by considering both activity level and financial contribution.